<a href="https://colab.research.google.com/github/iqra-1/agentic-ai-teaching-assistant/blob/main/AGENTIC_AI_TEACHING_ASSISTANT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AGENTIC AI TEACHING ASSISTANT
### Goal: Help students by understanding emails, checking records,
###       and drafting professional replies using an LLM.
### Uses: Google Gemini (LLM) + structured student data + simple tools
###[link text](https://) Output: Ready-to-send email replies (not JSON)

### 1. IMPORT LIBRARIES

In [ ]:
import re
from google import genai
from google.colab import userdata


In [ ]:
# Configure Gemini (Colab often auto-auths if you're signed in)
# If needed, get a free API key from: https://aistudio.google.com/
#GOOGLE_API_KEY = "YOUR_API_KEY_HERE"  # Optional in Colab if signed in


### 2. MOCK COURSE DATABASE (THE AGENT'S "MEMORY")


In [ ]:
# Simulates a student database (like a real LMS: Canvas, Moodle, etc.)
student_records = {
    "S1001": {
        "name": "Alice",
        "course": "CS101: Intro to Agentic AI",
        "assignments": {
            "A1": {"title": "LLM Basics", "due": "Oct 10, 2025", "submitted": True, "grade": 92},
            "A2": {"title": "Build an AI Agent", "due": "Oct 25, 2025", "submitted": False, "grade": None}
        },
        "instructor": "Dr. Smith",
        "resources": [
            "https://course.example.com/lectures",
            "https://colab.research.google.com/github/yourname/agent-lab"
        ]
    },
    "S1002": {
        "name": "Bob",
        "course": "CS101: Intro to Agentic AI",
        "assignments": {
            "A1": {"title": "LLM Basics", "due": "Oct 10, 2025", "submitted": False, "grade": None}
        },
        "instructor": "Dr. Smith",
        "resources": ["https://course.example.com/lectures"]
    }
}

# Course-wide policies (the agent must follow these)
COURSE_POLICY = """
- Late submissions: Accepted up to 48 hours late with 10% penalty.
- Grading: Released within 7 days after the deadline.
- Extensions: Require a valid reason (e.g., medical note) and must be approved by the instructor.
- Questions about grades: Email the instructor directly after 7 days.
"""


### 3. AGENT TOOLS (SIMULATED FUNCTIONS)
#### These represent the agent's "abilities"

In [ ]:
def extract_student_id(email_text):
    """
    Tool: Extract student ID from email (e.g., S1001).
    Uses regex to find patterns like S1234.
    """
    match = re.search(r'[Ss](\d{4,})', email_text)  # Matches S1001, s2025, etc.
    return f"S{match.group(1)}" if match else None


def extract_assignment_id(email_text):
    """
    Tool: Extract assignment ID (e.g., A1, A2).
    Simple version: looks for A followed by digits.
    """
    match = re.search(r'[Aa](\d+)', email_text)
    return f"A{match.group(1)}" if match else None


def lookup_student(student_id):
    """
    Tool: Query the student database (like a real LMS API).
    Returns student record or None.
    """
    return student_records.get(student_id)

### 4. AGENT BRAIN: DRAFT REPLY USING LLM
#### This is where the LLM acts as the agent's reasoning & communication engine

In [ ]:
def draft_reply(email_text, student_info):
    """
    Generate a ready-to-send email reply using Gemini LLM.
    The LLM is given:
      - The student's original email
      - Verified student data (from tools)
      - Course policy (to prevent hallucination)
    Output: Natural, professional, accurate email text.
    """
    # Detect student intent to guide the response
    text_lower = email_text.lower()
    if any(kw in text_lower for kw in ["grade", "score", "marked", "feedback"]):
        intent = "grade"
    elif any(kw in text_lower for kw in ["due", "deadline", "when", "submit"]):
        intent = "deadline"
    elif any(kw in text_lower for kw in ["resource", "link", "slide", "material", "note"]):
        intent = "resource"
    elif any(kw in text_lower for kw in ["help", "stuck", "confused", "don't understand"]):
        intent = "support"
    else:
        intent = "general"

    # Build context for the LLM
    if not student_info:
        # No student found → ask for ID
        prompt = f"""
        You are an AI Teaching Assistant for an online course.
        The student sent this email: "{email_text}"
        But no student record was found.
        Politely ask them to include their student ID (e.g., S1001).
        Keep it friendly and helpful.
        """
    else:
        name = student_info["name"]
        course = student_info["course"]
        assignments = student_info["assignments"]

        # Handle specific intents
        if intent == "deadline":
            aid = extract_assignment_id(email_text)
            if aid and aid in assignments:
                due_date = assignments[aid]["due"]
                prompt = f"""
                You are an AI Teaching Assistant for "{course}".
                Student {name} asked about the deadline for assignment {aid}.
                Verified info: It is due on {due_date}.
                Course policy: {COURSE_POLICY}
                Draft a kind, clear reply with the due date and offer help.
                Do NOT invent new info.
                """
            else:
                all_assignments = ", ".join(assignments.keys())
                prompt = f"""
                You are an AI Teaching Assistant for "{course}".
                Student {name} asked about an assignment deadline, but the assignment wasn't clear.
                Available assignments: {all_assignments}.
                Politely list them and ask which one they mean.
                """

        elif intent == "grade":
            aid = extract_assignment_id(email_text)
            if aid and aid in assignments and assignments[aid]["grade"] is not None:
                grade = assignments[aid]["grade"]
                prompt = f"""
                You are an AI Teaching Assistant for "{course}".
                Student {name} asked about their grade for {aid}.
                Verified grade: {grade}/100.
                Remind them that detailed feedback is in the LMS.
                Keep tone encouraging.
                """
            else:
                prompt = f"""
                You are an AI Teaching Assistant for "{course}".
                Student {name} asked about a grade, but either the assignment isn't graded yet or doesn't exist.
                Explain that grades are released within 7 days of the deadline.
                Offer to check again later.
                """

        elif intent == "resource":
            links = "\n".join(f"- {link}" for link in student_info["resources"])
            prompt = f"""
            You are an AI Teaching Assistant for "{course}".
            Student {name} asked for course resources.
            Verified links:
            {links}
            Share these links in a friendly way. Do not add extra links.
            """

        else:  # general or support
            prompt = f"""
            You are an AI Teaching Assistant for "{course}".
            Student {name} sent this message: "{email_text}"
            Course policy: {COURSE_POLICY}
            Offer general support: explain what you can help with (deadlines, grades, resources).
            If they seem stressed, be empathetic.
            Always remind them to include their student ID in future emails.
            """

    # CALL THE LLM (Gemini)
    client = genai.Client(api_key = userdata.get('GOOGLE_API_KEY'))
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text


### 5. MAIN AGENT WORKFLOW
#### Simulates the full agentic loop: perceive → reason → act

In [ ]:
def process_student_email(email_text):
    """
    Full agent pipeline:
      1. Extract student ID
      2. Look up student record
      3. Generate reply using LLM + tools
      4. Return ready-to-send email
    """
    print("Received student email:")
    print(f"   \"{email_text}\"")
    print("\n Agent is processing...\n")

    # Step 1: Extract student ID (perception)
    student_id = extract_student_id(email_text)

    # Step 2: Retrieve student data (tool use)
    student_info = lookup_student(student_id) if student_id else None

    # Step 3: Generate reply (reasoning + action via LLM)
    reply = draft_reply(email_text, student_info)

    return reply

### 6. DEMO: TEST THE AGENT
#### Run these examples to see your agent in action!

In [ ]:
print("="*60)
print("🎓 AGENTIC AI TEACHING ASSISTANT - DEMO")
print("="*60)

# Test Case 1: Deadline query (happy path)
email1 = "Hi, I'm Alice (S1001). When is A2 due? I'm a bit behind 😅"
reply1 = process_student_email(email1)
print("✅ REPLY:\n")
print(reply1)
print("\n" + "-"*60 + "\n")

# Test Case 2: Grade request
email2 = "Hello, this is Bob (S1002). Did I get my grade for A1 yet?"
reply2 = process_student_email(email2)
print("✅ REPLY:\n")
print(reply2)
print("\n" + "-"*60 + "\n")

# Test Case 3: Missing student ID
email3 = "I didn't get the lecture slides. Can you resend them?"
reply3 = process_student_email(email3)
print("✅ REPLY:\n")
print(reply3)
print("\n" + "-"*60 + "\n")

# Test Case 4: Emotional support + policy
email4 = "I'm overwhelmed with A2 and might miss the deadline. What can I do? - Alice (S1001)"
reply4 = process_student_email(email4)
print("✅ REPLY:\n")
print(reply4)

🎓 AGENTIC AI TEACHING ASSISTANT - DEMO
Received student email:
   "Hi, I'm Alice (S1001). When is A2 due? I'm a bit behind 😅"

 Agent is processing...

✅ REPLY:

Hi Alice,

Thanks for reaching out!

Assignment A2 is due on **October 25, 2025**.

Please let me know if you have any other questions about the assignment or anything else!

Best,

Your CS101 Teaching Assistant

------------------------------------------------------------

Received student email:
   "Hello, this is Bob (S1002). Did I get my grade for A1 yet?"

 Agent is processing...

✅ REPLY:

Hello Bob,

Thanks for reaching out!

Generally, grades for assignments are released within **7 days of the submission deadline**. We're working hard to get everything graded accurately.

If you'd like, I can check again for you later in the week once we're past that grading window. Just let me know which specific assignment you're referring to, and I can keep an eye out for it.

--------------------------------------------------------

###7. HUMAN-IN-THE-LOOP (OPTIONAL)
#### human review before sending

In [ ]:
print("\n📤 Ready to send? [Simulated human review]")
approval = input("Type 'send' to approve: ")
if approval.strip().lower() in ['send', 'yes']:
    print("✅ Email sent!")
else:
    print("✏️ Draft saved for editing.")


📤 Ready to send? [Simulated human review]
Type 'send' to approve: send
✅ Email sent!
